# 경기도 공공차량 이송 환자 응급의료기관 종별(권역 vs 지역) 이진 분류
**사용법**: 아래 셀 맨 위쪽 `FILE_PATH`만 본인 파일 경로로 바꾼 뒤 셀을 실행하세요. (Shift+Enter)

- 처음 실행하면 LightGBM/XGBoost/CatBoost 등이 자동 설치됩니다.
- 점검용으로 빨리 돌려 보려면 `QUICK_MODE = True` (최종 논문 결과는 반드시 `False`)
- 결과(그림 600 DPI PNG, 표 CSV, 요약 리포트)는 `ICT_outputs/` 폴더에 저장됩니다.

- 그림 스타일: 참고 NEDIS 노트북처럼 청록·차분한 주황 2색과 회색을 중심으로 정리하고, 격자선·채움·반복 주석을 줄였습니다.

In [ ]:
# -*- coding: utf-8 -*-
"""
==============================================================================
 경기도 공공차량(119구급차·경찰차 등) 이송 환자의
 최적 응급의료기관 종별(권역응급의료센터 vs 지역응급의료센터) 이진 분류
 -- ICT 학술대회 투고용 All-in-One 분석 코드 (Jupyter Notebook 한 셀 실행용) --
==============================================================================
 [실행 방법]
   1) 아래 [1단계]의 FILE_PATH 만 본인 파일 경로로 바꾼다.
   2) 이 코드를 Jupyter Notebook 셀 하나에 통째로 붙여넣고 실행한다.
   3) 결과물은 OUTPUT_DIR 폴더에 저장된다 (그림 PNG 600 DPI + 표 CSV + 요약 리포트).

 [단계 구성]
   1. 환경 설정  2. 데이터 로드 및 필터링  3. 파생변수 2종 생성 및 전처리
   4. EDA 시각화  5. 모델 학습 및 Ablation Study  6. SHAP XAI 분석
   7. 논문 작성용 핵심 요약 리포트
==============================================================================
"""

# =============================================================================
# [1단계] 환경 설정 : 라이브러리 자동 설치 / 경로·옵션 / 한글 폰트 / 논문용 그래프 스타일
# =============================================================================
import sys, os, subprocess, importlib, importlib.util, warnings, time, re, glob, shutil, platform

# ---------------------------------------------------------------------------
# 1-1. 사용자 설정 (여기만 고치면 됩니다)
# ---------------------------------------------------------------------------
FILE_PATH    = "C:/Users/sojjo/Desktop/DSJA/3. 빅데이터 시각화/NEDIS_응급표본자료_2020_2024.xlsx"   # ★ 본인 파일 경로 (xlsx 또는 csv)
SHEET_NAME   = "데이터셋"                      # 엑셀일 때 시트 이름 (없으면 첫 번째 시트 사용)
OUTPUT_DIR   = "ICT_outputs"                  # 그림/표/리포트가 저장될 폴더

RANDOM_STATE = 42       # 재현성을 위한 난수 시드 (같은 값이면 항상 같은 결과)
TEST_SIZE    = 0.2      # 학습:검증 = 8:2
N_SPLITS     = 5        # 5-Fold 교차검증
DPI          = 600      # 저장 해상도 (학술대회/Nature 투고용)
FIG_LANG     = "auto"   # 그림 글자: "ko"(한글) / "en"(영어) / "auto"(한글 폰트 있으면 한글, 없으면 영어)
N_JOBS       = -1       # 사용할 CPU 코어 수 (-1 = 전부)
N_BOOTSTRAP  = 500      # 95% 신뢰구간용 부트스트랩 반복 횟수
SHAP_MAX_ROWS = 6000    # SHAP 계산에 쓸 검증(Test) 표본 최대 개수
QUICK_MODE   = False    # True: 빠른 점검용(트리 수·반복 횟수 축소). 최종 논문 결과는 반드시 False 로!
SHOW_FIGURES = True     # 노트북 화면에 그림 표시 여부

DURATION_MAX_HOURS = 24 * 365   # 고통지속도 상한(1년 초과는 입력 오류로 간주 → 결측 처리)
DATE_ORIGIN  = "2020-01-01"     # 날짜를 '경과일'로 바꿀 때의 기준일
MIN_N_FOR_RATE = 100            # 3번 그림에서 비율을 보여줄 최소 환자 수(너무 적은 범주는 불안정)

# 선택 사항: UMLS 코드를 사람이 읽는 이름으로 바꾸고 싶을 때 {'C0004604': 'Back pain', ...} 형태로 채우세요.
UMLS_NAME_MAP = {}

# ---------------------------------------------------------------------------
# 1-2. 라이브러리 자동 설치 (설치돼 있으면 건너뜀)
# ---------------------------------------------------------------------------
def _pip_install(spec):
    """pip 설치. 일부 환경(PEP668)에서는 --break-system-packages 가 필요해서 두 번 시도."""
    for extra in ([], ["--break-system-packages"]):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec, *extra],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return True
        except Exception:
            continue
    return False

_REQUIRED = {            # import 이름 : pip 이름
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "openpyxl": "openpyxl",
    "sklearn": "scikit-learn>=1.2",
    "lightgbm": "lightgbm", "xgboost": "xgboost", "catboost": "catboost",
}
for _imp, _pip in _REQUIRED.items():
    if importlib.util.find_spec(_imp) is None:
        print(f"[설치] {_pip} 설치 중...")
        if not _pip_install(_pip):
            print(f"  ⚠ {_pip} 자동 설치 실패 → 터미널에서 직접 `pip install {_pip}` 해 주세요.")
importlib.invalidate_caches()

# scikit-learn 버전이 너무 낮으면(OneHotEncoder 옵션 때문에) 업그레이드
try:
    import sklearn
    if tuple(int(v) for v in sklearn.__version__.split(".")[:2]) < (1, 2):
        _pip_install("scikit-learn>=1.2"); importlib.invalidate_caches()
except Exception:
    pass

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, roc_curve, precision_recall_curve)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)
IN_NOTEBOOK = "ipykernel" in sys.modules
T0 = time.time()

def log(msg=""):
    """경과 시간(분:초)과 함께 진행 상황을 출력"""
    s = int(time.time() - T0)
    print(f"[{s//60:02d}:{s%60:02d}] {msg}", flush=True)

def banner(title):
    print("\n" + "=" * 100 + f"\n {title}\n" + "=" * 100, flush=True)

def show_table(df, title=None, max_rows=60):
    """노트북에서는 예쁜 표로, 일반 파이썬에서는 텍스트로 출력"""
    if title: print(f"\n▶ {title}")
    d = df if len(df) <= max_rows else df.head(max_rows)
    if IN_NOTEBOOK:
        from IPython.display import display
        display(d)
    else:
        print(d.to_string())
    if len(df) > max_rows:
        print(f"   ... (총 {len(df)}행 중 {max_rows}행만 표시, 전체는 CSV 파일 참고)")

def save_table(df, name, index=True):
    """표를 CSV(엑셀에서 한글 안 깨지는 utf-8-sig)로 저장"""
    path = os.path.join(OUTPUT_DIR, name)
    df.to_csv(path, index=index, encoding="utf-8-sig")
    return path

# ---------------------------------------------------------------------------
# 1-3. 한글 폰트 설정 (글자 깨짐 방지)
#      윈도우(맑은 고딕) → 맥(AppleGothic) → 리눅스/코랩(나눔/노토) → 자동 설치 → 그래도 없으면 영어 그림
# ---------------------------------------------------------------------------
def setup_korean_font():
    cands = ["Malgun Gothic", "NanumGothic", "NanumBarunGothic", "AppleGothic", "Apple SD Gothic Neo",
             "Noto Sans CJK KR", "Noto Sans KR", "Pretendard", "UnDotum", "Baekmuk Gulim"]

    def _find():
        names = {f.name for f in font_manager.fontManager.ttflist}
        for c in cands:
            if c in names:
                return c
        return None

    found = _find()
    if found:
        return found
    # (a) 리눅스/코랩: 나눔 폰트 설치 시도
    try:
        if platform.system() == "Linux" and shutil.which("apt-get") and hasattr(os, "geteuid") and os.geteuid() == 0:
            subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL, timeout=180)
            for p in glob.glob("/usr/share/fonts/truetype/nanum/*.ttf"):
                font_manager.fontManager.addfont(p)
            found = _find()
            if found:
                return found
    except Exception:
        pass
    # (b) 어디서나 되는 방법: koreanize-matplotlib (나눔고딕 포함 패키지)
    try:
        if importlib.util.find_spec("koreanize_matplotlib") is None:
            _pip_install("koreanize-matplotlib"); importlib.invalidate_caches()
        import koreanize_matplotlib  # noqa: F401  (import 만으로 폰트 등록)
        found = _find()
        if found:
            return found
    except Exception:
        pass
    return None

KO_FONT = setup_korean_font()
if FIG_LANG == "auto":
    LANG = "ko" if KO_FONT else "en"
else:
    LANG = FIG_LANG
    if LANG == "ko" and not KO_FONT:
        print("⚠ 한글 폰트를 찾지 못해 그림 글자를 영어로 전환합니다.")
        LANG = "en"

# ---------------------------------------------------------------------------
# 1-4. 논문용 그래프 설정: 참고 NEDIS 노트북의 절제된 시각 스타일
#      청록·주황 2색 + 중성 회색, 흰 배경, 최소한의 축/주석
# ---------------------------------------------------------------------------
W1, W2 = 3.46, 7.09          # 단폭 88mm / 양단폭 180mm (인치)

# 두 집단을 구분하는 핵심 색상만 사용합니다.
INK = "#26333b"
INK2 = "#65737a"
GRID = "#e7ecee"
BLUE = "#16728b"             # 청록: 지역센터 / 주요 강조
ORANGE = "#ce8144"           # 차분한 주황: 권역센터 / 보조 강조
PALE_BLUE = "#eef5f7"
MID_GREY = "#9aa6ab"
LIGHT_GREY = "#c7cfd2"

C_REG, C_LOC = ORANGE, BLUE

MODEL_NAMES = ["Logistic Regression", "Random Forest", "LightGBM", "XGBoost", "CatBoost"]
# 모델 비교에서는 청록·주황만 강조하고 나머지는 회색 계열로 눌러 색상 과밀을 줄입니다.
MODEL_COL = {
    "Logistic Regression": "#8e9a9f",
    "Random Forest": "#b8c1c4",
    "LightGBM": "#586970",
    "XGBoost": ORANGE,
    "CatBoost": BLUE,
}
MODEL_LS = {
    "Logistic Regression": "-",
    "Random Forest": "--",
    "LightGBM": "-.",
    "XGBoost": ":",
    "CatBoost": "-",
}
VARIANT_COL = {
    "umls_only": BLUE,
    "serial_only": ORANGE,
    "both": "#586970",
    "none": "#b8c1c4",
}
DIV_CMAP = LinearSegmentedColormap.from_list(
    "quiet_diverging", [BLUE, "#f5f6f5", ORANGE]
)
SHAP_CMAP = LinearSegmentedColormap.from_list(
    "quiet_shap", ["#9dbbc4", "#edf0ef", "#d5a078"]
)
SEQ_CMAP = LinearSegmentedColormap.from_list(
    "quiet_sequential", [PALE_BLUE, "#c2d9df", BLUE]
)

_en_font = next((f for f in ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"]
                 if f in {x.name for x in font_manager.fontManager.ttflist}), "DejaVu Sans")
_FONT = [KO_FONT, "DejaVu Sans"] if (LANG == "ko" and KO_FONT) else [_en_font, "DejaVu Sans"]

mpl.rcParams.update({
    "font.family": _FONT,
    "font.size": 7.5,
    "axes.labelsize": 7.5,
    "axes.titlesize": 8,
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.6,
    "xtick.labelsize": 6.5,
    "ytick.labelsize": 6.5,
    "legend.fontsize": 6.5,
    "legend.frameon": False,
    "legend.handlelength": 1.4,
    "legend.handletextpad": 0.45,
    "legend.borderaxespad": 0.35,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "axes.unicode_minus": False,
    "figure.dpi": 120,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.08,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "lines.linewidth": 1.0,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.facecolor": "white",
    "text.color": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK2,
    "ytick.color": INK2,
    "axes.edgecolor": INK2,
})

# ---------------------------------------------------------------------------
# 1-5. 그림 글자(한글/영어) 번역 도우미
# ---------------------------------------------------------------------------
VAR_EN = {
    "응급의료기관 지역": "ED region", "응급의료기관 종별": "ED level", "내원일자": "Visit date",
    "내원시간": "Visit time", "연령": "Age group", "성별": "Sex", "환자주소지역": "Residence",
    "발병일자": "Onset date", "발병시간": "Onset time", "질병여부": "Disease / non-disease",
    "의도성 여부": "Intent", "손상기전": "Injury mechanism", "내원경로": "Referral route",
    "내원수단": "Arrival mode", "주증상-1": "Chief complaint (UMLS)", "주증상-1 일련번호": "Chief complaint serial no.",
    "응급증상 해당 여부": "Emergency symptom", "고통지속도": "Onset-to-arrival time",
    "발병시간_범주": "Onset time band", "권역센터(Y)": "Regional EMC (Y)",
}
CAT_EN = {
    "권역응급의료센터": "Regional EMC", "지역응급의료센터": "Local EMC", "권역센터": "Regional", "지역센터": "Local",
    "119구급차": "119 ambulance", "경찰차 등 공공차량": "Police/public vehicle",
    "남성": "Male", "여성": "Female", "비의도적 사고": "Unintentional", "자해/자살": "Self-harm/suicide",
    "폭력/타살": "Violence/homicide", "기타": "Other", "미상": "Unknown", "UNKNOWN": "Unknown",
    "자동차교통사고": "Car crash", "자전거교통사고": "Bicycle crash", "오토바이교통사고": "Motorcycle crash",
    "기타교통사고": "Other traffic", "미상교통사고": "Unknown traffic", "추락": "Fall", "미끄러짐": "Slip",
    "부딪힘": "Blunt/collision", "베임/찔림": "Cut/pierce", "기계": "Machinery", "화상": "Burn", "익수": "Drowning",
    "중독": "Poisoning", "질식/목맴": "Suffocation/hanging", "기타(성폭력,감전 등)": "Other (assault, shock)",
    "직접내원": "Direct visit", "외부에서 전원": "Transfer-in", "외래에서 의뢰": "Outpatient referral",
    "심야(00~05시)": "Late night (00-05)", "오전(06~11시)": "Morning (06-11)", "오후(12~17시)": "Afternoon (12-17)",
    "야간(18~23시)": "Evening (18-23)", "서울시": "Seoul", "부산시": "Busan", "대구시": "Daegu", "인천시": "Incheon",
    "광주시": "Gwangju", "대전시": "Daejeon", "울산시": "Ulsan", "경기도": "Gyeonggi", "강원도": "Gangwon",
    "충청북도": "Chungbuk", "충청남도/세종": "Chungnam/Sejong", "전라북도": "Jeonbuk", "전라남도": "Jeonnam",
    "경상북도": "Gyeongbuk", "경상남도": "Gyeongnam", "제주도": "Jeju", "의료기관 구급차": "Hospital ambulance",
    "기타 구급차": "Other ambulance", "항공이송": "Air transport", "질병": "Disease", "질병 외": "Non-disease",
}
def tr(ko, en):      return ko if LANG == "ko" else en
def V(name):         return name if LANG == "ko" else VAR_EN.get(name, name)
def C(val):          return str(val) if LANG == "ko" else CAT_EN.get(str(val), str(val))

# ---------------------------------------------------------------------------
# 1-6. 그림 저장/표시 도우미
# ---------------------------------------------------------------------------
FIG_LIST = []
def save_fig(fig, name):
    """600 DPI PNG 로 저장하고 노트북에 표시"""
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=DPI, facecolor="white")
    FIG_LIST.append(path)
    log(f"그림 저장: {path}")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)

def panel_label(ax, letter, x=-0.10, y=1.06):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=9.5, fontweight="bold",
            va="bottom", ha="right", color=INK)

def soft_grid(ax, axis="y"):
    # 참고 그림처럼 배경 격자는 제거하고 축과 데이터만 남깁니다.
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.tick_params(length=2.5, width=0.6)

def wilson_ci(k, n, z=1.96):
    """비율의 95% Wilson 신뢰구간 (표본이 작아도 안정적)"""
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = np.where(n > 0, k / np.maximum(n, 1), np.nan)
    den = 1 + z**2 / np.maximum(n, 1)
    c = (p + z**2 / (2 * np.maximum(n, 1))) / den
    h = z * np.sqrt(p * (1 - p) / np.maximum(n, 1) + z**2 / (4 * np.maximum(n, 1)**2)) / den
    return c - h, c + h

def pfmt(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def pstr(p):
    return "p<0.001" if p < 0.001 else f"p={p:.3f}"

log(f"환경 설정 완료 | 그림 언어={LANG} | 폰트={KO_FONT if LANG=='ko' else _en_font} | QUICK_MODE={QUICK_MODE}")
log(f"버전: pandas {pd.__version__}, sklearn {sklearn.__version__}, lightgbm {lgb.__version__}, "
    f"xgboost {xgb.__version__}, catboost {importlib.import_module('catboost').__version__}")


# =============================================================================
# [2단계] 데이터 로드 및 필터링
# =============================================================================
banner("[2단계] 데이터 로드 및 필터링")

# ---------------------------------------------------------------------------
# 2-1. 파일 읽기 (xlsx / csv 모두 지원)
# ---------------------------------------------------------------------------
def load_table(path, sheet):
    if not os.path.exists(path):
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}\n→ [1단계]의 FILE_PATH 를 확인해 주세요.")
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xlsm", ".xls"):
        xl = pd.ExcelFile(path)
        sh = sheet if sheet in xl.sheet_names else xl.sheet_names[0]
        log(f"엑셀 시트 '{sh}' 읽는 중 (시트 목록: {xl.sheet_names})")
        return xl.parse(sh)
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            continue
    raise ValueError("CSV 인코딩을 읽지 못했습니다.")

raw = load_table(FILE_PATH, SHEET_NAME)
log(f"원본 데이터: {raw.shape[0]:,}행 × {raw.shape[1]}열")

# ---------------------------------------------------------------------------
# 2-2. 컬럼명 통일
#      💡 쉬운 설명: 실제 엑셀에는 'PTMIEMCL' 같은 영문 코드명이 들어 있습니다.
#         코드북을 보고 우리가 쓸 한글 변수명으로 바꿔 줍니다. (이미 한글이면 그대로 둡니다.)
# ---------------------------------------------------------------------------
RAW_COLS = ["응급의료기관 지역", "응급의료기관 종별", "내원일자", "내원시간", "연령", "성별", "환자주소지역",
            "발병일자", "발병시간", "질병여부", "의도성 여부", "손상기전", "내원경로", "내원수단",
            "주증상-1", "주증상-1 일련번호", "응급증상 해당 여부"]            # 보유 원본 변수 17개
CODE2KO = {"PTMIEMAR": "응급의료기관 지역", "PTMIEMCL": "응급의료기관 종별", "PTMIINDT": "내원일자",
           "PTMIINTM": "내원시간", "PTMIBRTD": "연령", "PTMISEXX": "성별", "PTMIPTAR": "환자주소지역",
           "PTMIAKDT": "발병일자", "PTMIAKTM": "발병시간", "PTMIDGKD": "질병여부", "PTMIARCF": "의도성 여부",
           "PTMIARCS": "손상기전", "PTMIINRT": "내원경로", "PTMIINMN": "내원수단", "PTMIMNSY": "주증상-1",
           "PTMIMSSR": "주증상-1 일련번호", "PTMIEMSY": "응급증상 해당 여부"}
_canon = {k.replace(" ", ""): k for k in RAW_COLS}
_canon.update({"응급기관종별": "응급의료기관 종별", "의도성여부": "의도성 여부", "응급증상해당여부": "응급증상 해당 여부"})
_rename = {}
for c in raw.columns:
    key = str(c).strip()
    if key.upper() in CODE2KO:
        _rename[c] = CODE2KO[key.upper()]
    elif key.replace(" ", "") in _canon:
        _rename[c] = _canon[key.replace(" ", "")]
raw = raw.rename(columns=_rename)
_missing = [c for c in RAW_COLS if c not in raw.columns]
if _missing:
    raise KeyError(f"필요한 변수가 없습니다: {_missing}\n현재 컬럼: {list(raw.columns)}")
df0 = raw[RAW_COLS].copy()

# ---------------------------------------------------------------------------
# 2-3. 코드 → 한글 라벨 디코딩 (코드북 기준)
#      ※ 성별: 코드북은 1=남성, 2=여성 입니다. (요청서의 '1=여성'은 코드북과 반대라 코드북을 따름)
#         → 외상 환자는 남성이 더 많은데 데이터에서도 1이 55%라서 코드북이 맞다고 판단했습니다.
#         → 반대로 쓰고 싶으면 아래 '성별' 줄의 매핑만 바꾸세요.
# ---------------------------------------------------------------------------
REGION = {1: "서울시", 2: "부산시", 3: "대구시", 4: "인천시", 5: "광주시", 6: "대전시", 7: "울산시", 8: "경기도",
          9: "강원도", 10: "충청북도", 11: "충청남도/세종", 12: "전라북도", 13: "전라남도", 14: "경상북도",
          15: "경상남도", 16: "제주도", 99: "미상"}
CODEBOOK = {
    "응급의료기관 지역": REGION,
    "응급의료기관 종별": {1: "권역응급의료센터", 2: "지역응급의료센터"},
    "성별": {1: "남성", 2: "여성"},
    "환자주소지역": REGION,
    "질병여부": {1: "질병", 2: "질병 외"},
    "의도성 여부": {1: "비의도적 사고", 2: "자해/자살", 3: "폭력/타살", 8: "기타", 9: "미상"},
    "손상기전": {1: "자동차교통사고", 3: "자전거교통사고", 4: "오토바이교통사고", 8: "기타교통사고",
              9: "미상교통사고", 10: "추락", 11: "미끄러짐", 20: "부딪힘", 30: "베임/찔림", 40: "기계",
              50: "화상", 60: "익수", 70: "중독", 80: "질식/목맴", 88: "기타(성폭력,감전 등)", 99: "미상"},
    "내원경로": {1: "직접내원", 2: "외부에서 전원", 3: "외래에서 의뢰", 8: "기타", 9: "미상"},
    "내원수단": {1: "119구급차", 2: "의료기관 구급차", 3: "기타 구급차", 4: "경찰차 등 공공차량", 5: "항공이송"},
}
# 연령은 5세 단위 '연령군 코드(1~17)'. 순서가 의미 있는 값이라 숫자 그대로 사용하고, 그림에는 라벨을 붙입니다.
AGE_LABELS = {1: "0-4", 2: "5-9", 3: "10-14", 4: "15-19", 5: "20-24", 6: "25-29", 7: "30-34", 8: "35-39",
              9: "40-44", 10: "45-49", 11: "50-54", 12: "55-59", 13: "60-64", 14: "65-69", 15: "70-74",
              16: "75-79", 17: "80+"}      # (코드북의 '75-59세'는 오타로 보고 75-79세로 처리)

def decode_series(s, mapping):
    """숫자 코드면 라벨로 바꾸고, 이미 한글 라벨이면 그대로 둠"""
    num = pd.to_numeric(s, errors="coerce")
    mapped = num.map(mapping)
    return s.astype(object).where(mapped.isna(), mapped)

df0["연령"] = pd.to_numeric(df0["연령"], errors="coerce")
for col, mp in CODEBOOK.items():
    df0[col] = decode_series(df0[col], mp)
# 응급증상 해당 여부는 Y/N 문자 그대로 사용
df0["응급증상 해당 여부"] = df0["응급증상 해당 여부"].astype(str).str.strip().str.upper().replace({"NAN": np.nan})

# ---------------------------------------------------------------------------
# 2-4. 필터링 (3가지 조건) + 단계별 인원 기록
# ---------------------------------------------------------------------------
funnel = [("전체 원본", len(df0))]
m1 = df0["응급의료기관 지역"] == "경기도";                                       d1 = df0[m1]
funnel.append(("1) 응급의료기관 지역 = 경기도", len(d1)))
m2 = d1["응급의료기관 종별"].isin(["권역응급의료센터", "지역응급의료센터"]);       d2 = d1[m2]
funnel.append(("2) 종별 ∈ {권역, 지역 응급의료센터}", len(d2)))
m3 = d2["내원수단"].isin(["119구급차", "경찰차 등 공공차량"]);                     df = d2[m3].copy().reset_index(drop=True)
funnel.append(("3) 내원수단 ∈ {119구급차, 경찰차 등 공공차량}", len(df)))
funnel_df = pd.DataFrame(funnel, columns=["단계", "남은 환자 수"])
show_table(funnel_df, "필터링 단계별 환자 수")
save_table(funnel_df, "Table0_filter_funnel.csv", index=False)
if len(df) == 0:
    raise ValueError("필터 후 데이터가 0건입니다. 코드북/컬럼 값을 확인해 주세요.")

# ---------------------------------------------------------------------------
# 2-5. 타겟 변수(Y) : 권역응급의료센터 = 1, 지역응급의료센터 = 0
# ---------------------------------------------------------------------------
df["Y"] = (df["응급의료기관 종별"] == "권역응급의료센터").astype(int)
log(f"최종 분석 대상 {len(df):,}명 | 권역(1) {df['Y'].sum():,}명 ({df['Y'].mean()*100:.1f}%) | "
    f"지역(0) {(1-df['Y']).sum():,.0f}명 ({(1-df['Y'].mean())*100:.1f}%)")


# =============================================================================
# [3단계] 파생변수 2종 생성 및 전처리
#   ① 고통지속도  : 발병 → 응급실 도착까지 걸린 총 시간(시간 단위)
#   ② 발병시간_범주 : 발병 시각을 시간대(심야/오전/오후/야간/미상)로 분류
#   ※ 요청대로 이 2개 외의 파생변수는 만들지 않습니다.
# =============================================================================
banner("[3단계] 파생변수 2종 생성 및 전처리")

# ---------------------------------------------------------------------------
# 3-1. 날짜·시간 파싱 도우미
#      💡 쉬운 설명: '발병일자 20200124' + '발병시간 1120' 을 합쳐 '2020-01-24 11:20' 이라는
#         진짜 시각으로 바꿔야 날짜가 달라도(예: 어제 밤 발병 → 오늘 아침 도착) 정확히 뺄셈이 됩니다.
# ---------------------------------------------------------------------------
UNKNOWN_DATE = {"11111111", "99999999", "1111", "9999", "0", "00000000", "nan", "NaN", "None", "", "<NA>"}
UNKNOWN_TIME_ONSET = {1111, 9999}          # 발병시간의 '미상' 코드 (요청 규칙: 1111, 9999 → NaN)

def _clean_str(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

def parse_date(s):
    """YYYYMMDD(숫자/문자/날짜형) → datetime. 미상 코드·잘못된 날짜는 NaN"""
    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s)
    t = _clean_str(s)
    t = t.mask(t.isin(UNKNOWN_DATE))
    out = pd.to_datetime(t, format="%Y%m%d", errors="coerce")
    rest = out.isna() & t.notna()                         # 'YYYY-MM-DD' 같은 다른 형식 대비
    if rest.any():
        out[rest] = pd.to_datetime(t[rest], errors="coerce")
    return out

def parse_minutes(s, unknown=()):
    """HHMM(숫자/문자/'HH:MM') → 자정 기준 분(0~1439). 미상 코드·범위 밖 값은 NaN"""
    t = _clean_str(s).str.replace(":", "", regex=False)
    num = pd.to_numeric(t, errors="coerce")
    if unknown:
        num = num.mask(num.isin(list(unknown)))
    hh, mm = np.floor(num / 100), num % 100
    ok = num.notna() & (hh >= 0) & (hh <= 23) & (mm >= 0) & (mm <= 59)
    return (hh * 60 + mm).where(ok)

# ---------------------------------------------------------------------------
# 3-2. 파생변수 ① 고통지속도 (단위: 시간)
# ---------------------------------------------------------------------------
onset_date  = parse_date(df["발병일자"])
arrive_date = parse_date(df["내원일자"])
onset_min   = parse_minutes(df["발병시간"], unknown=UNKNOWN_TIME_ONSET)
arrive_min  = parse_minutes(df["내원시간"])                 # 내원시간의 1111 은 실제 값일 수 있어 그대로 사용

onset_dt  = onset_date  + pd.to_timedelta(onset_min,  unit="m")     # 날짜 + 시간 → 발병일시
arrive_dt = arrive_date + pd.to_timedelta(arrive_min, unit="m")     # 날짜 + 시간 → 내원일시
dur_h = (arrive_dt - onset_dt).dt.total_seconds() / 3600            # (내원일시 - 발병일시) → 시간

n_unknown = int(dur_h.isna().sum())
n_neg = int((dur_h < 0).sum())                                      # 발병이 도착보다 늦다? → 명백한 입력 오류
n_big = int((dur_h > DURATION_MAX_HOURS).sum())                     # 1년 초과 → 입력 오류로 간주
dur_h = dur_h.mask((dur_h < 0) | (dur_h > DURATION_MAX_HOURS))      # 오류치는 NaN (삭제하지 않고 결측 처리)
df["고통지속도"] = dur_h
log(f"고통지속도 생성 | 미상코드로 인한 결측 {n_unknown:,}건, 음수 오류 {n_neg:,}건, "
    f"{DURATION_MAX_HOURS/24:.0f}일 초과 오류 {n_big:,}건 → 모두 NaN | 최종 유효값 {int(dur_h.notna().sum()):,}건")

# ---------------------------------------------------------------------------
# 3-3. 파생변수 ② 발병시간_범주
# ---------------------------------------------------------------------------
onset_hour = np.floor(onset_min / 60)
def _band(h):
    if pd.isna(h):  return "미상"
    h = int(h)
    if h <= 5:      return "심야(00~05시)"
    if h <= 11:     return "오전(06~11시)"
    if h <= 17:     return "오후(12~17시)"
    return "야간(18~23시)"
BAND_ORDER = ["심야(00~05시)", "오전(06~11시)", "오후(12~17시)", "야간(18~23시)", "미상"]
df["발병시간_범주"] = onset_hour.map(_band)
log("발병시간_범주 분포: " + ", ".join(f"{k} {v:,}" for k, v in df["발병시간_범주"].value_counts().reindex(BAND_ORDER).items()))

# ---------------------------------------------------------------------------
# 3-4. 모델 입력 변수(X) 구성
#      - 내원일자/발병일자 : '2020-01-01 이후 경과일' 숫자 1개씩 (모델이 읽을 수 있게 변환)
#      - 내원시간/발병시간 : HHMM(예 1152)을 '소수 시간'(11.87)으로 변환 (1159→1200 의 점프 제거)
#      - 연령 : 연령군 코드(1~17) 그대로 (순서가 있는 숫자)
#      - 주증상-1(UMLS) / 주증상-1 일련번호 : 둘 다 문자(str) 범주형, 결측은 'UNKNOWN'
#      - 손상기전 / 의도성 여부 : '미상'을 지우지 않고 하나의 범주로 보존
# ---------------------------------------------------------------------------
origin = pd.Timestamp(DATE_ORIGIN)
X_full = pd.DataFrame(index=df.index)
X_full["연령"]     = pd.to_numeric(df["연령"], errors="coerce")
X_full["내원일자"] = (arrive_date - origin).dt.days.astype(float)
X_full["내원시간"] = arrive_min / 60.0
X_full["발병일자"] = (onset_date - origin).dt.days.astype(float)
X_full["발병시간"] = onset_min / 60.0
X_full["고통지속도"] = df["고통지속도"]
NUM_FEATURES = ["연령", "내원일자", "내원시간", "발병일자", "발병시간", "고통지속도"]

def to_cat_str(s, unknown_token="UNKNOWN"):
    t = s.astype(object).where(s.notna(), None)
    t = t.map(lambda v: unknown_token if v is None else re.sub(r"\.0$", "", str(v).strip()))
    return t.replace({"": unknown_token, "-": unknown_token, "nan": unknown_token, "NaN": unknown_token,
                      "None": unknown_token, "<NA>": unknown_token})

CAT_CANDIDATES = ["응급의료기관 지역", "성별", "환자주소지역", "질병여부", "의도성 여부", "손상기전", "내원경로", "내원수단",
                  "주증상-1", "주증상-1 일련번호", "응급증상 해당 여부", "발병시간_범주"]
for c in CAT_CANDIDATES:
    X_full[c] = to_cat_str(df[c])
X_full["주증상-1"] = X_full["주증상-1"].str.upper()

# 값이 하나뿐인 변수는 정보가 없으므로 자동 제외 (응급의료기관 지역 = 경기도 단일값, 질병여부 등)
CONST_DROPPED = [c for c in X_full.columns if X_full[c].nunique(dropna=False) <= 1]
for c in CONST_DROPPED:
    log(f"단일값 변수 자동 제외: '{c}' (값 = {X_full[c].iloc[0]!r})")
CAT_FEATURES = [c for c in CAT_CANDIDATES if c not in CONST_DROPPED]
FEATURES = NUM_FEATURES + CAT_FEATURES
X_full = X_full[FEATURES]
for c in CAT_FEATURES:                         # 트리 모델이 쓸 수 있도록 category 타입으로 (범주 목록은 전체 기준 고정)
    X_full[c] = X_full[c].astype("category")
y_full = df["Y"].values
log(f"입력 변수 X: 총 {len(FEATURES)}개 = 수치형 {len(NUM_FEATURES)} + 범주형 {len(CAT_FEATURES)}")
log(f"  수치형: {NUM_FEATURES}")
log(f"  범주형: {CAT_FEATURES}")
log(f"  주증상-1(UMLS) 고유값 {X_full['주증상-1'].nunique():,}개 | 주증상-1 일련번호 고유값 {X_full['주증상-1 일련번호'].nunique():,}개")

# ---------------------------------------------------------------------------
# 3-5. 데이터 품질 요약표 (결측 비율 등)
# ---------------------------------------------------------------------------
dq = pd.DataFrame({
    "유형": ["수치"] * len(NUM_FEATURES) + ["범주"] * len(CAT_FEATURES),
    "결측(NaN) 수": [int(X_full[c].isna().sum()) for c in FEATURES],
    "'미상/UNKNOWN' 수": [0] * len(NUM_FEATURES) + [int((X_full[c].astype(str).isin(["미상", "UNKNOWN", "미상교통사고"])).sum()) for c in CAT_FEATURES],
    "고유값 수": [int(X_full[c].nunique()) for c in FEATURES],
}, index=FEATURES)
dq["결측 비율(%)"] = (dq["결측(NaN) 수"] / len(X_full) * 100).round(2)
show_table(dq, "입력 변수 품질 요약 (고통지속도·발병일자·발병시간의 결측은 '미상 코드'에서 발생)")
save_table(dq, "Table0_data_quality.csv")

# ---------------------------------------------------------------------------
# 3-6. 학습/검증 분할 (층화 8:2) -- 이후 모든 전처리는 '학습 데이터'에서만 학습(누수 방지)
# ---------------------------------------------------------------------------
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, stratify=y_full, random_state=RANDOM_STATE)
X_tr_all, X_te_all = X_full.iloc[idx_tr].reset_index(drop=True), X_full.iloc[idx_te].reset_index(drop=True)
y_tr, y_te = y_full[idx_tr], y_full[idx_te]
log(f"분할: Train {len(idx_tr):,}명 (권역 {y_tr.mean()*100:.1f}%) / Test {len(idx_te):,}명 (권역 {y_te.mean()*100:.1f}%)")


# =============================================================================
# [4단계] EDA (탐색적 데이터 분석) 및 600 DPI 시각화
# =============================================================================
banner("[4단계] EDA 시각화")
df["종별_라벨"] = np.where(df["Y"] == 1, "권역센터", "지역센터")
OVERALL_RATE = df["Y"].mean()

# ---------------------------------------------------------------------------
# 4-0. Table 1 : 권역 vs 지역 환자 특성 비교표 (논문 첫 번째 표)
# ---------------------------------------------------------------------------
def build_table1():
    rows = []
    g1, g0 = df[df["Y"] == 1], df[df["Y"] == 0]
    rows.append(["N", "", f"{len(g1):,}", f"{len(g0):,}", ""])
    # (a) 수치형: 중앙값 [Q1–Q3], Mann-Whitney U 검정
    num_vars = {"고통지속도": "고통지속도 (시간)", "내원시간": "내원시각 (시)", "발병시간": "발병시각 (시)"}
    tmp = pd.DataFrame({"고통지속도": df["고통지속도"], "내원시간": X_full["내원시간"], "발병시간": X_full["발병시간"], "Y": df["Y"]})
    for c, nm in num_vars.items():
        a, b = tmp.loc[tmp.Y == 1, c].dropna(), tmp.loc[tmp.Y == 0, c].dropna()
        p = stats.mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan
        f = lambda s: f"{s.median():.2f} [{s.quantile(.25):.2f}–{s.quantile(.75):.2f}]"
        rows.append([nm, "중앙값 [Q1–Q3]", f(a), f(b), pfmt(p)])
    # (b) 범주형: n (%) , 카이제곱 검정
    cat_vars = ["연령", "성별", "환자주소지역", "의도성 여부", "손상기전", "내원경로", "내원수단", "응급증상 해당 여부", "발병시간_범주"]
    for c in cat_vars:
        s = df[c].map(AGE_LABELS) if c == "연령" else df[c].astype(str)
        ct = pd.crosstab(s, df["Y"])
        if 0 not in ct.columns: ct[0] = 0
        if 1 not in ct.columns: ct[1] = 0
        p = stats.chi2_contingency(ct[[1, 0]])[1] if ct.shape[0] > 1 else np.nan
        first = True
        order = [AGE_LABELS[k] for k in sorted(AGE_LABELS)] if c == "연령" else (BAND_ORDER if c == "발병시간_범주" else ct.sum(1).sort_values(ascending=False).index)
        for lv in order:
            if lv not in ct.index: continue
            n1, n0 = ct.loc[lv, 1], ct.loc[lv, 0]
            rows.append([c if first else "", str(lv), f"{n1:,} ({n1/len(g1)*100:.1f}%)", f"{n0:,} ({n0/len(g0)*100:.1f}%)", pfmt(p) if first else ""])
            first = False
    return pd.DataFrame(rows, columns=["변수", "범주/통계", "권역응급의료센터", "지역응급의료센터", "p-value"])
table1 = build_table1()
save_table(table1, "Table1_baseline_characteristics.csv", index=False)
show_table(table1[table1["p-value"] != ""][["변수", "권역응급의료센터", "지역응급의료센터", "p-value"]]
           .rename(columns={"변수": "변수(p-value가 있는 행만 요약)"}), "Table 1 요약 (전체 표는 Table1_baseline_characteristics.csv)")

# ---------------------------------------------------------------------------
# 4-1. [EDA 1] 타겟 분포 + 내원수단별 교차표  → Fig1
# ---------------------------------------------------------------------------
ct_cnt = pd.crosstab(df["내원수단"], df["종별_라벨"])[["권역센터", "지역센터"]]
ct_cnt["합계"] = ct_cnt.sum(1)
ct_pct = (ct_cnt[["권역센터", "지역센터"]].div(ct_cnt["합계"], axis=0) * 100).round(1)
ct_show = pd.concat([ct_cnt, ct_pct.add_suffix(" (행%)")], axis=1)
ct_show.loc["전체"] = [ct_cnt["권역센터"].sum(), ct_cnt["지역센터"].sum(), ct_cnt["합계"].sum(), round(OVERALL_RATE * 100, 1), round((1 - OVERALL_RATE) * 100, 1)]
show_table(ct_show, "타겟 × 내원수단 교차표")
save_table(ct_show, "Table2_target_by_arrival_mode.csv")

ct2 = pd.crosstab(df["내원수단"], df["Y"])
chi2, p_chi, _, _ = stats.chi2_contingency(ct2)
cramers_v = np.sqrt(chi2 / ct2.values.sum())
a_, b_, c_, d_ = ct2.loc["119구급차", 1], ct2.loc["119구급차", 0], ct2.loc["경찰차 등 공공차량", 1], ct2.loc["경찰차 등 공공차량", 0]
OR_ = (a_ * d_) / (b_ * c_)
se_ = np.sqrt(1 / a_ + 1 / b_ + 1 / c_ + 1 / d_)
OR_CI = (np.exp(np.log(OR_) - 1.96 * se_), np.exp(np.log(OR_) + 1.96 * se_))
log(f"내원수단 × 종별 카이제곱 {pstr(p_chi)}, Cramér's V={cramers_v:.3f}, 119구급차 대비 경찰차 등 OR(권역)={1/OR_:.2f} "
    f"(95% CI {1/OR_CI[1]:.2f}–{1/OR_CI[0]:.2f})")

fig, axes = plt.subplots(1, 2, figsize=(W2, 2.5), gridspec_kw={"width_ratios": [1, 1.5], "wspace": 0.55})
ax = axes[0]
vals = [int((df["Y"] == 0).sum()), int((df["Y"] == 1).sum())]
bars = ax.bar([0, 1], vals, color=[C_LOC, C_REG], width=0.62, edgecolor="none")
for xi, v in zip([0, 1], vals):
    ax.text(xi, v + max(vals) * 0.015, f"{v:,}\n({v/sum(vals)*100:.1f}%)", ha="center", va="bottom", fontsize=6.5)
ax.set_xticks([0, 1]); ax.set_xticklabels([C("지역센터"), C("권역센터")])
ax.set_ylabel(tr("환자 수 (명)", "Patients (n)")); ax.set_ylim(0, max(vals) * 1.22); soft_grid(ax)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_title(tr("타겟 클래스 분포", "Target class distribution"), loc="left", fontsize=7.5); panel_label(ax, "a")
ax = axes[1]
modes = ["119구급차", "경찰차 등 공공차량"]
for i, mname in enumerate(modes):
    r_loc, r_reg = ct_pct.loc[mname, "지역센터"], ct_pct.loc[mname, "권역센터"]
    ax.barh(i, r_loc, color=C_LOC, height=0.58, edgecolor="none", label=C("지역센터") if i == 0 else None)
    ax.barh(i, r_reg, left=r_loc, color=C_REG, height=0.58, edgecolor="none", label=C("권역센터") if i == 0 else None)
    ax.text(r_loc / 2, i, f"{r_loc:.1f}%", ha="center", va="center", color="white", fontsize=6.5)
    ax.text(r_loc + r_reg / 2, i, f"{r_reg:.1f}%", ha="center", va="center", color="white", fontsize=6.5)
ax.set_yticks(range(2)); ax.set_yticklabels([f"{C(m)}\n(n={int(ct_cnt.loc[m, '합계']):,})" for m in modes])
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel(tr("행 기준 비율 (%)", "Row percentage (%)"))
ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.2), handlelength=1, columnspacing=1.2)
ax.set_title(f"χ² p {('<0.001' if p_chi < 0.001 else '= ' + f'{p_chi:.3f}')}, Cramér's V = {cramers_v:.2f}", loc="right", fontsize=6, color=INK2)
panel_label(ax, "b", x=-0.3)
save_fig(fig, "Fig1_target_distribution_by_arrival_mode.png")

# ---------------------------------------------------------------------------
# 4-2. [EDA 2] 고통지속도 기초통계량 + 권역 vs 지역 비교 분포  → Fig2
# ---------------------------------------------------------------------------
def describe_dur(s):
    s = s.dropna()
    return pd.Series({"N(유효)": len(s), "평균": s.mean(), "표준편차": s.std(), "최소": s.min(), "Q1(25%)": s.quantile(.25),
                      "중앙값": s.median(), "Q3(75%)": s.quantile(.75), "P95": s.quantile(.95), "최대": s.max()})
dur_stats = pd.DataFrame({"전체": describe_dur(df["고통지속도"]),
                          "권역센터": describe_dur(df.loc[df.Y == 1, "고통지속도"]),
                          "지역센터": describe_dur(df.loc[df.Y == 0, "고통지속도"])}).T.round(2)
show_table(dur_stats, "고통지속도(시간) 기초통계량")
save_table(dur_stats, "Table3_onset_to_arrival_stats.csv")
da, db = df.loc[df.Y == 1, "고통지속도"].dropna(), df.loc[df.Y == 0, "고통지속도"].dropna()
U, p_mw = stats.mannwhitneyu(da, db)
eff = U / (len(da) * len(db))             # '권역 환자가 지역 환자보다 더 길 확률' (0.5면 차이 없음)
log(f"고통지속도 권역 vs 지역 Mann–Whitney U {pstr(p_mw)}, 효과크기 P(권역>지역)={eff:.3f}")

FLOOR_H = 1 / 60.0                         # 로그 변환을 위해 1분 미만은 1분으로 보정(그림에서만 사용)
fig, axes = plt.subplots(1, 2, figsize=(W2, 2.7), gridspec_kw={"wspace": 0.35})
tick_h = [1 / 6, 1, 6, 24, 168]; tick_lb = [tr("10분", "10 min"), tr("1시간", "1 h"), tr("6시간", "6 h"), tr("1일", "1 d"), tr("1주", "1 wk")]
ax = axes[0]
data_box = [np.log10(np.clip(db, FLOOR_H, None)), np.log10(np.clip(da, FLOOR_H, None))]
bp = ax.boxplot(data_box, positions=[0, 1], widths=0.5, patch_artist=True, showfliers=False, whis=(5, 95),
                medianprops=dict(color=INK, linewidth=1.2), whiskerprops=dict(color=INK2, linewidth=0.7),
                capprops=dict(color=INK2, linewidth=0.7), boxprops=dict(linewidth=0.7, edgecolor=INK2))
for patch, col in zip(bp["boxes"], [C_LOC, C_REG]):
    patch.set_facecolor(col); patch.set_alpha(0.22)
ax.set_xticks([0, 1]); ax.set_xticklabels([C("지역센터"), C("권역센터")])
ax.set_yticks(np.log10(tick_h)); ax.set_yticklabels(tick_lb)
ax.set_ylim(min(np.log10(1 / 6) - 0.25, np.log10(max(min(da.quantile(.05), db.quantile(.05)), FLOOR_H)) - 0.12),
            max(np.log10(max(da.quantile(.95), db.quantile(.95))) + 0.55, np.log10(24) + 0.2))
ax.set_ylabel(tr("고통지속도 (로그 축)", "Onset-to-arrival time (log scale)")); soft_grid(ax)
ax.set_title(tr("상자그림 (수염 5–95 백분위)", "Box plot (whiskers 5th–95th pct)"), loc="left", fontsize=7.5); panel_label(ax, "a")
ax = axes[1]
xs_max = np.log10(max(da.quantile(.995), db.quantile(.995)))
for s, col, nm in [(db, C_LOC, "지역센터"), (da, C_REG, "권역센터")]:
    sns.kdeplot(np.log10(np.clip(s, FLOOR_H, None)), ax=ax, color=col, fill=False, linewidth=1.15, label=C(nm), clip=(np.log10(FLOOR_H), xs_max), cut=0, bw_adjust=1.0)
    ax.axvline(np.log10(max(s.median(), FLOOR_H)), color=col, linestyle=":", linewidth=0.7, alpha=0.9)
ax.set_xticks(np.log10(tick_h)); ax.set_xticklabels(tick_lb); ax.set_xlim(np.log10(FLOOR_H), xs_max)
ax.set_xlabel(tr("고통지속도 (로그 축)", "Onset-to-arrival time (log scale)")); ax.set_ylabel(tr("밀도", "Density"))
ax.legend(loc="upper right"); soft_grid(ax)
ax.text(0.98, 0.72, f"Mann–Whitney p {('<0.001' if p_mw < 0.001 else '= ' + f'{p_mw:.3f}')}", transform=ax.transAxes, ha="right", fontsize=6, color=INK2)
ax.set_title(tr("밀도 곡선 (점선=중앙값)", "KDE (dashed = median)"), loc="left", fontsize=7.5); panel_label(ax, "b")
save_fig(fig, "Fig2_onset_to_arrival_time_distribution.png")

# ---------------------------------------------------------------------------
# 4-3. [EDA 3] 발병시간_범주·손상기전·의도성 여부별 권역센터 이송 비율 (Wilson 95% CI)  → Fig3
# ---------------------------------------------------------------------------
def rate_table(col, order=None, min_n=0):
    g = df.groupby(col)["Y"].agg(["sum", "count"]).rename(columns={"sum": "k", "count": "n"})
    g = g[g["n"] >= min_n]
    g["rate"] = g["k"] / g["n"] * 100
    g["lo"], g["hi"] = [v * 100 for v in wilson_ci(g["k"].values, g["n"].values)]
    if order is not None:
        g = g.reindex([o for o in order if o in g.index])
    else:
        g = g.sort_values("rate", ascending=False)
    return g

def rate_bars(ax, g, title, letter, show_ylabel=True):
    ypos = np.arange(len(g))
    ax.barh(ypos, g["rate"], color=BLUE, alpha=0.92, height=0.58, edgecolor="none")
    ax.errorbar(g["rate"], ypos, xerr=[g["rate"] - g["lo"], g["hi"] - g["rate"]], fmt="none", ecolor=INK2, elinewidth=0.7, capsize=1.5)
    for yi, (r, hi, n) in enumerate(zip(g["rate"], g["hi"], g["n"])):
        ax.text(hi + 0.8, yi, f"{r:.1f}%", va="center", fontsize=6.2, color=INK2)
    ax.axvline(OVERALL_RATE * 100, color=INK2, linestyle=":", linewidth=0.7)
    ax.set_yticks(ypos); ax.set_yticklabels([C(i) for i in g.index]); ax.invert_yaxis()
    ax.set_xlim(0, max(g["hi"].max() * 1.55, OVERALL_RATE * 100 * 1.3)); ax.set_xlabel(tr("권역센터 이송 비율 (%)", "Regional EMC transport rate (%)"))
    ax.set_title(title, loc="left", fontsize=7.5); soft_grid(ax, "x"); panel_label(ax, letter, x=-0.02 if False else -0.1)

g_band = rate_table("발병시간_범주", BAND_ORDER)
g_mech = rate_table("손상기전", min_n=MIN_N_FOR_RATE)
g_int = rate_table("의도성 여부", min_n=MIN_N_FOR_RATE)
show_table(pd.concat({"발병시간_범주": g_band, "손상기전": g_mech, "의도성 여부": g_int}).round(2), "범주별 권역센터 이송 비율 (rate=%, lo~hi=95% CI)")
save_table(pd.concat({"발병시간_범주": g_band, "손상기전": g_mech, "의도성 여부": g_int}).round(3), "Table4_rate_by_category.csv")
fig = plt.figure(figsize=(W2, 3.4))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.25], height_ratios=[len(g_band) + 0.8, len(g_int) + 1.6], wspace=0.75, hspace=0.62)
ax1 = fig.add_subplot(gs[0, 0]); ax3 = fig.add_subplot(gs[1, 0]); ax2 = fig.add_subplot(gs[:, 1])
rate_bars(ax1, g_band, tr("발병시간_범주", "Onset time band"), "a")
rate_bars(ax3, g_int, tr("의도성 여부", "Intent"), "c")
rate_bars(ax2, g_mech, tr(f"손상기전 (n≥{MIN_N_FOR_RATE})", f"Injury mechanism (n≥{MIN_N_FOR_RATE})"), "b")
fig.text(0.5, -0.01, tr(f"점선 = 전체 평균 {OVERALL_RATE*100:.1f}%, 오차막대 = 95% Wilson 신뢰구간", f"Dashed line = overall mean {OVERALL_RATE*100:.1f}%; error bars = 95% Wilson CI"), ha="center", fontsize=6, color=INK2)
save_fig(fig, "Fig3_regional_rate_by_onset_band_mechanism_intent.png")

# ---------------------------------------------------------------------------
# 4-4. [EDA 4] 주요 수치형 변수 상관관계 히트맵 (Spearman)  → Fig4
#      💡 쉬운 설명: 두 변수가 같이 오르내리는 정도(-1~+1). 순서형·치우친 분포에 강한 Spearman 사용.
# ---------------------------------------------------------------------------
corr_df = pd.DataFrame({
    "연령": X_full["연령"], "성별": (df["성별"] == "남성").astype(float).where(df["성별"].isin(["남성", "여성"])),
    "내원일자": X_full["내원일자"], "내원시간": X_full["내원시간"], "발병일자": X_full["발병일자"], "발병시간": X_full["발병시간"],
    "고통지속도": X_full["고통지속도"], "응급증상 해당 여부": (df["응급증상 해당 여부"] == "Y").astype(float).where(df["응급증상 해당 여부"].isin(["Y", "N"])),
    "권역센터(Y)": df["Y"].astype(float)})
corr = corr_df.corr(method="spearman")
corr_lab = {"성별": tr("성별(남=1)", "Sex (male=1)"), "응급증상 해당 여부": tr("응급증상(Y=1)", "Emergency sx (Y=1)")}
names = [corr_lab.get(c, V(c)) for c in corr.columns]
save_table(corr.round(3), "Table5_spearman_correlation.csv")
fig, ax = plt.subplots(figsize=(W2 * 0.72, W2 * 0.62))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap=DIV_CMAP, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", annot_kws={"fontsize": 6},
            linewidths=0.4, linecolor="white", square=True, ax=ax, xticklabels=names, yticklabels=names,
            cbar_kws={"shrink": 0.7, "aspect": 18, "label": tr("Spearman 상관계수 ρ", "Spearman ρ")})
ax.tick_params(axis="x", rotation=40, labelsize=6.5); ax.tick_params(axis="y", rotation=0, labelsize=6.5)
for l in ax.get_xticklabels(): l.set_ha("right")
ax.collections[0].colorbar.outline.set_linewidth(0.4); ax.collections[0].colorbar.ax.tick_params(labelsize=6)
save_fig(fig, "Fig4_correlation_heatmap.png")


# =============================================================================
# [5단계] 모델 학습 및 Ablation Study
# =============================================================================
banner("[5단계] 모델 학습 · 5-Fold CV · Ablation Study")

# ---------------------------------------------------------------------------
# 5-1. 평가지표 함수
#      💡 쉬운 설명 (권역센터 = '양성', 1)
#         Recall(민감도) = 실제 권역센터가 필요한 환자 중 모델이 '권역'이라고 맞춘 비율 → 놓치면 위험하므로 가장 중요
#         Precision      = 모델이 '권역'이라고 한 환자 중 실제 권역이 맞은 비율 → 낮으면 권역센터 과밀(불필요한 이송)
#         Specificity    = 지역센터로 충분한 환자를 '지역'으로 맞춘 비율
#         ROC-AUC/PR-AUC = 기준점(임계값)과 무관하게 모델이 두 집단을 얼마나 잘 가르는가 (1에 가까울수록 좋음)
# ---------------------------------------------------------------------------
def evaluate(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"Accuracy": accuracy_score(y, pred), "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0), "Specificity": tn / max(tn + fp, 1),
            "F1": f1_score(y, pred, zero_division=0), "ROC_AUC": roc_auc_score(y, p),
            "PR_AUC": average_precision_score(y, p), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
METRICS = ["Accuracy", "Precision", "Recall", "Specificity", "F1", "ROC_AUC", "PR_AUC"]

# ---------------------------------------------------------------------------
# 5-2. 하이퍼파라미터 (클래스 불균형 보정 포함)
#      - 불균형 보정: LR/RF/LightGBM → class_weight='balanced', XGBoost → scale_pos_weight, CatBoost → auto_class_weights='Balanced'
#      - 부스팅 3종은 '조기 종료(early stopping)'로 과적합을 막음 (학습 데이터의 10%를 내부 검증용으로 사용)
# ---------------------------------------------------------------------------
MAX_TREES = 150 if QUICK_MODE else 1500
RF_TREES = 100 if QUICK_MODE else 300
if QUICK_MODE:
    N_BOOTSTRAP = min(N_BOOTSTRAP, 100); SHAP_MAX_ROWS = min(SHAP_MAX_ROWS, 2000)
POS_WEIGHT = float((y_tr == 0).sum() / (y_tr == 1).sum())     # 음성/양성 비 → XGBoost scale_pos_weight

class Fitted:
    """학습된 모델 + 전처리를 한 묶음으로 관리 (예측 시 같은 전처리를 자동 적용)"""
    def __init__(self, kind, model, cols, cat_cols, num_cols, best_iter=None):
        self.kind, self.model, self.cols, self.cat_cols, self.num_cols, self.best_iter = kind, model, cols, cat_cols, num_cols, best_iter
    def prep(self, X):
        X = X[self.cols]
        if self.kind == "CatBoost":            # CatBoost 는 범주형을 문자열로 직접 받음 (cat_features 네이티브 처리)
            X = X.copy()
            for c in self.cat_cols: X[c] = X[c].astype(str)
        return X
    def predict_proba1(self, X):
        return self.model.predict_proba(self.prep(X))[:, 1]

def _sk_preprocessor(num_cols, cat_cols, scale):
    # scale=True → 로지스틱 회귀용(희소행렬, 희귀 범주 30명 미만은 '기타'로 묶음)
    # scale=False → 랜덤포레스트용(밀집행렬이 훨씬 빠름, 희귀 범주 50명 미만은 '기타'로 묶어 열 수를 줄임)
    ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30 if scale else 50,
                        sparse_output=bool(scale), dtype=np.float32)
    tfs = []
    num_main = [c for c in num_cols if not (scale and c == "고통지속도")]
    if num_main:
        steps = [("imp", SimpleImputer(strategy="median"))] + ([("sc", StandardScaler())] if scale else [])
        tfs.append(("num", Pipeline(steps), num_main))
    if scale and "고통지속도" in num_cols:      # 로지스틱 회귀: 심하게 치우친 고통지속도는 log1p 후 표준화
        tfs.append(("dur", Pipeline([("imp", SimpleImputer(strategy="median")), ("log", FunctionTransformer(np.log1p)),
                                     ("sc", StandardScaler())]), ["고통지속도"]))
    if cat_cols:
        tfs.append(("cat", ohe, cat_cols))
    return ColumnTransformer(tfs, remainder="drop", sparse_threshold=0.3 if scale else 0)

def fit_model(kind, X, y, cols, seed=RANDOM_STATE):
    cat_cols = [c for c in cols if c in CAT_FEATURES]; num_cols = [c for c in cols if c in NUM_FEATURES]
    X = X[cols]
    if kind == "Logistic Regression":
        m = Pipeline([("pre", _sk_preprocessor(num_cols, cat_cols, scale=True)),
                      ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=300, tol=1e-3, solver="lbfgs"))])
        m.fit(X, y); return Fitted(kind, m, cols, cat_cols, num_cols)
    if kind == "Random Forest":
        m = Pipeline([("pre", _sk_preprocessor(num_cols, cat_cols, scale=False)),
                      ("clf", RandomForestClassifier(n_estimators=RF_TREES, min_samples_leaf=3, max_features="sqrt",
                                                     class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=seed))])
        m.fit(X, y); return Fitted(kind, m, cols, cat_cols, num_cols)
    # --- 부스팅 3종: 학습 데이터 10%를 조기 종료용 검증셋으로 분리 ---
    Xa, Xv, ya, yv = train_test_split(X, y, test_size=0.1, stratify=y, random_state=seed)
    if kind == "LightGBM":
        m = lgb.LGBMClassifier(n_estimators=MAX_TREES, learning_rate=0.05, num_leaves=48, min_child_samples=30,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=2.0,
                               class_weight="balanced", cat_smooth=20, cat_l2=10, min_data_per_group=50,
                               max_cat_to_onehot=4, random_state=seed, n_jobs=N_JOBS, verbose=-1)
        m.fit(Xa, ya, eval_set=[(Xv, yv)], eval_metric="auc",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        return Fitted(kind, m, cols, cat_cols, num_cols, m.best_iteration_)
    if kind == "XGBoost":
        m = xgb.XGBClassifier(n_estimators=MAX_TREES, learning_rate=0.05, max_depth=6, min_child_weight=3,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, scale_pos_weight=POS_WEIGHT,
                              tree_method="hist", enable_categorical=True, max_cat_to_onehot=4,
                              eval_metric="auc", early_stopping_rounds=50, n_jobs=N_JOBS, random_state=seed, verbosity=0)
        m.fit(Xa, ya, eval_set=[(Xv, yv)], verbose=False)
        return Fitted(kind, m, cols, cat_cols, num_cols, m.best_iteration)
    if kind == "CatBoost":
        tmp = Fitted(kind, None, cols, cat_cols, num_cols)
        Xa_c, Xv_c = tmp.prep(Xa), tmp.prep(Xv)
        m = CatBoostClassifier(iterations=MAX_TREES, learning_rate=0.1, depth=6, l2_leaf_reg=3,
                               auto_class_weights="Balanced", eval_metric="AUC", od_type="Iter", od_wait=50,
                               random_seed=seed, verbose=0, thread_count=N_JOBS if N_JOBS > 0 else -1,
                               allow_writing_files=False)
        m.fit(Xa_c, ya, cat_features=cat_cols, eval_set=(Xv_c, yv), use_best_model=True)
        return Fitted(kind, m, cols, cat_cols, num_cols, m.get_best_iteration())
    raise ValueError(kind)

# ---------------------------------------------------------------------------
# 5-3. Ablation 설계: 주증상-1(UMLS) vs 주증상-1 일련번호 vs 둘 다
#      'none'(둘 다 제외)은 요청 3종과 비교하기 위한 참고용 기준선입니다.
# ---------------------------------------------------------------------------
UMLS, SERIAL = "주증상-1", "주증상-1 일련번호"
VARIANTS = {
    "umls_only":   [c for c in FEATURES if c != SERIAL],
    "serial_only": [c for c in FEATURES if c != UMLS],
    "both":        list(FEATURES),                                   # ← 본 실험(메인 모델)
    "none":        [c for c in FEATURES if c not in (UMLS, SERIAL)], # 참고용 기준선
}
VARIANT_LABEL = {"umls_only": tr("UMLS 코드만", "UMLS code only"), "serial_only": tr("일련번호만", "Serial no. only"),
                 "both": tr("둘 다", "Both"), "none": tr("(참고) 둘 다 제외", "(ref.) Neither")}

# ---------------------------------------------------------------------------
# 5-4. 5-Fold CV (학습 데이터 안에서) + 학습 전체로 재학습 후 Test 평가
#      💡 쉬운 설명: 학습 데이터를 5등분해 4/5로 학습·1/5로 점검을 5번 반복 → 평균±표준편차로 '안정성'을 확인.
#         그 후 한 번도 안 본 Test 20%로 최종 성능을 잰다. (CV 예측은 임계값 선택에도 사용: Test 엿보기 방지)
# ---------------------------------------------------------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(np.zeros(len(y_tr)), y_tr))
RES = {}      # RES[variant][model] = {...}

def run_experiment(variant, kind, keep_model=False):
    cols = VARIANTS[variant]
    t = time.time()
    fold_rows, oof = [], np.zeros(len(y_tr))
    for k, (a, b) in enumerate(FOLDS):
        f = fit_model(kind, X_tr_all.iloc[a], y_tr[a], cols)
        p = f.predict_proba1(X_tr_all.iloc[b]); oof[b] = p
        fold_rows.append(evaluate(y_tr[b], p))
    cv = pd.DataFrame(fold_rows)
    f_final = fit_model(kind, X_tr_all, y_tr, cols)
    p_te = f_final.predict_proba1(X_te_all)
    te = evaluate(y_te, p_te)
    RES.setdefault(variant, {})[kind] = {"cv": cv, "oof": oof, "test": te, "p_test": p_te,
                                         "fitted": f_final if keep_model else None, "sec": time.time() - t}
    log(f"  [{variant:11s}] {kind:19s} | CV AUC {cv['ROC_AUC'].mean():.4f}±{cv['ROC_AUC'].std():.4f} | "
        f"Test AUC {te['ROC_AUC']:.4f}, F1 {te['F1']:.4f}, Recall {te['Recall']:.4f} | {time.time()-t:.0f}s")

log("① 본 실험: 5개 모델 × (5-Fold CV + Test)  [feature = 전체 변수(주증상 2종 모두 포함)]")
for kind in MODEL_NAMES:
    run_experiment("both", kind, keep_model=True)

log("② Ablation: 주증상 변수 구성만 바꿔서 같은 5개 모델 재실험")
for variant in ["umls_only", "serial_only", "none"]:
    for kind in MODEL_NAMES:
        run_experiment(variant, kind)

# ---------------------------------------------------------------------------
# 5-5. 결과 표 만들기
# ---------------------------------------------------------------------------
def ms(series, d=3):
    return f"{series.mean():.{d}f} ± {series.std():.{d}f}"

main_rows, main_cv_rows = [], []
for kind in MODEL_NAMES:
    r = RES["both"][kind]
    row = {"Model": kind}
    for m in METRICS: row[m] = r["test"][m]
    main_rows.append(row)
    main_cv_rows.append({"Model": kind, **{m: ms(r["cv"][m]) for m in METRICS}})
test_tbl = pd.DataFrame(main_rows).set_index("Model")
cv_tbl = pd.DataFrame(main_cv_rows).set_index("Model")
show_table(cv_tbl, f"[본 실험] {N_SPLITS}-Fold CV 성능 (평균 ± 표준편차, 학습 데이터 내부)")
show_table(test_tbl.round(4), "[본 실험] Test(검증 20%) 성능  ※ 권역센터 Recall(민감도) 강조")
save_table(cv_tbl, "Table6_main_cv_performance.csv"); save_table(test_tbl.round(4), "Table7_main_test_performance.csv")

# 부트스트랩 95% 신뢰구간 (Test 세트 재표집)
def bootstrap_ci(y, p, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); n = len(y); out = {"ROC_AUC": [], "PR_AUC": [], "F1": [], "Recall": []}
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() == y[i].max(): continue
        out["ROC_AUC"].append(roc_auc_score(y[i], p[i])); out["PR_AUC"].append(average_precision_score(y[i], p[i]))
        pr = (p[i] >= 0.5).astype(int)
        out["F1"].append(f1_score(y[i], pr, zero_division=0)); out["Recall"].append(recall_score(y[i], pr, zero_division=0))
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in out.items()}
log(f"부트스트랩 95% CI 계산 중 (반복 {N_BOOTSTRAP}회 × 5모델)...")
BOOT = {k: bootstrap_ci(y_te, RES["both"][k]["p_test"]) for k in MODEL_NAMES}
ci_tbl = pd.DataFrame({k: {m: f"{RES['both'][k]['test'][m]:.3f} ({BOOT[k][m][0]:.3f}–{BOOT[k][m][1]:.3f})" for m in ["ROC_AUC", "PR_AUC", "F1", "Recall"]} for k in MODEL_NAMES}).T
show_table(ci_tbl, "[본 실험] Test 성능 95% 부트스트랩 신뢰구간")
save_table(ci_tbl, "Table8_main_test_bootstrap_CI.csv")

# 최고 성능 모델 선택 (CV ROC-AUC 기준 — Test 를 보고 고르면 '엿보기'가 되므로 CV 로 선택)
cv_auc = {k: RES["both"][k]["cv"]["ROC_AUC"].mean() for k in MODEL_NAMES}
BEST_MODEL = max(cv_auc, key=cv_auc.get)
SHAP_MODEL = max(["CatBoost", "XGBoost"], key=lambda k: cv_auc[k])
log(f"CV ROC-AUC 기준 최고 모델 = {BEST_MODEL} ({cv_auc[BEST_MODEL]:.4f}) | SHAP 분석 대상(CatBoost/XGBoost 중) = {SHAP_MODEL}")

# 최적 임계값: CV 예측(OOF)에서 F1 이 최대가 되는 값 (기본 0.5 와 비교용)
oof_b = RES["both"][BEST_MODEL]["oof"]
pr_, rc_, th_ = precision_recall_curve(y_tr, oof_b)
f1s = 2 * pr_[:-1] * rc_[:-1] / np.maximum(pr_[:-1] + rc_[:-1], 1e-12)
THR_OPT = float(th_[int(np.nanargmax(f1s))])
te_opt = evaluate(y_te, RES["both"][BEST_MODEL]["p_test"], THR_OPT)
log(f"{BEST_MODEL} 최적 임계값(OOF F1 최대) = {THR_OPT:.3f} → Test Recall {te_opt['Recall']:.3f}, Precision {te_opt['Precision']:.3f}, F1 {te_opt['F1']:.3f}")

# ---------------------------------------------------------------------------
# 5-6. Ablation 결과표 : 주증상-1(UMLS)만 vs 일련번호만 vs 둘 다
# ---------------------------------------------------------------------------
abl_rows = []
for v in ["umls_only", "serial_only", "both", "none"]:
    for kind in MODEL_NAMES:
        r = RES[v][kind]
        abl_rows.append({"Feature set": VARIANT_LABEL[v], "Model": kind,
                         "CV ROC-AUC": ms(r["cv"]["ROC_AUC"], 4), "CV F1": ms(r["cv"]["F1"]), "CV Recall": ms(r["cv"]["Recall"]),
                         "Test ROC-AUC": round(r["test"]["ROC_AUC"], 4), "Test F1": round(r["test"]["F1"], 4),
                         "Test Recall": round(r["test"]["Recall"], 4), "Test Precision": round(r["test"]["Precision"], 4)})
abl_tbl = pd.DataFrame(abl_rows)
save_table(abl_tbl, "Table9_ablation_full.csv", index=False)
show_table(abl_tbl, "[Ablation] 주증상-1(UMLS) vs 일련번호 vs 둘 다 (모든 모델)", max_rows=40)
# 모델별로 한눈에 보는 요약(Test ROC-AUC / F1 / Recall)
abl_wide = {}
for m_, nm in [("ROC_AUC", "ROC-AUC"), ("F1", "F1"), ("Recall", "Recall")]:
    abl_wide[nm] = pd.DataFrame({VARIANT_LABEL[v]: {k: RES[v][k]["test"][m_] for k in MODEL_NAMES} for v in ["umls_only", "serial_only", "both", "none"]}).round(4)
abl_summary = pd.concat(abl_wide, axis=1)
show_table(abl_summary, "[Ablation 요약] Test 지표 (행=모델, 열=지표×주증상 변수 구성)")
save_table(abl_summary, "Table9_ablation_summary_wide.csv")

# ---------------------------------------------------------------------------
# 5-7. 그림: ROC / PR 곡선 + 모델 비교 (Fig5)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.55), gridspec_kw={"wspace": 0.42, "width_ratios": [1, 1, 1.15]})
ax = axes[0]
for k in MODEL_NAMES:
    fpr, tpr, _ = roc_curve(y_te, RES["both"][k]["p_test"])
    ax.plot(fpr, tpr, color=MODEL_COL[k], linewidth=1.0, linestyle=MODEL_LS[k], label=f"{k} ({RES['both'][k]['test']['ROC_AUC']:.3f})")
ax.plot([0, 1], [0, 1], color=INK2, linestyle=":", linewidth=0.7)
ax.set_xlabel(tr("1 - 특이도 (False positive rate)", "False positive rate")); ax.set_ylabel(tr("민감도 (True positive rate)", "True positive rate"))
ax.set_title("ROC", loc="left", fontsize=7.5); ax.legend(loc="lower right", fontsize=5.7, handlelength=1.25); soft_grid(ax, "both"); panel_label(ax, "a")
ax.set_aspect("equal", adjustable="box")
ax = axes[1]
for k in MODEL_NAMES:
    pr, rc, _ = precision_recall_curve(y_te, RES["both"][k]["p_test"])
    ax.plot(rc, pr, color=MODEL_COL[k], linewidth=1.0, linestyle=MODEL_LS[k], label=f"{k} ({RES['both'][k]['test']['PR_AUC']:.3f})")
ax.axhline(y_te.mean(), color=INK2, linestyle=":", linewidth=0.7)
ax.text(0.5, y_te.mean() + 0.015, tr(f"무작위 기준 {y_te.mean():.2f}", f"Chance {y_te.mean():.2f}"), fontsize=5.5, color=INK2)
ax.set_xlabel(tr("재현율 (Recall)", "Recall")); ax.set_ylabel(tr("정밀도 (Precision)", "Precision")); ax.set_ylim(0, 1.02)
ax.set_title(tr("정밀도–재현율", "Precision–Recall"), loc="left", fontsize=7.5); ax.legend(loc="lower left", bbox_to_anchor=(0.0, 0.0), fontsize=5.7, handlelength=1.25); soft_grid(ax, "both"); panel_label(ax, "b")
ax = axes[2]
mets = [("ROC_AUC", "ROC-AUC"), ("PR_AUC", "PR-AUC"), ("F1", "F1"), ("Recall", tr("Recall(권역)", "Recall"))]
ypos = np.arange(len(mets)); off = np.linspace(-0.3, 0.3, len(MODEL_NAMES))
for j, k in enumerate(MODEL_NAMES):
    mu = [RES["both"][k]["cv"][m].mean() for m, _ in mets]; sd = [RES["both"][k]["cv"][m].std() for m, _ in mets]
    ax.errorbar(mu, ypos + off[j], xerr=sd, fmt="o", color=MODEL_COL[k], markersize=2.8, elinewidth=0.7, capsize=1.1)
ax.set_yticks(ypos); ax.set_yticklabels([n for _, n in mets]); ax.invert_yaxis(); ax.set_xlabel(tr(f"{N_SPLITS}-Fold CV 평균 ± SD", f"{N_SPLITS}-fold CV mean ± SD"))
ax.set_title(tr("교차검증 성능", "Cross-validated performance"), loc="left", fontsize=7.5); soft_grid(ax, "x"); panel_label(ax, "c", x=-0.28)
save_fig(fig, "Fig5_model_comparison_ROC_PR_CV.png")

# ---------------------------------------------------------------------------
# 5-8. 그림: Confusion Matrix (5개 모델 + 최고 모델 최적 임계값)  (Fig6)
# ---------------------------------------------------------------------------
def draw_cm(ax, tn, fp, fn, tp, title, letter):
    cm = np.array([[tn, fp], [fn, tp]]); rown = cm / cm.sum(1, keepdims=True)
    ax.imshow(rown, cmap=SEQ_CMAP, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}\n({rown[i, j]*100:.1f}%)", ha="center", va="center", fontsize=6.3,
                    color="white" if rown[i, j] > 0.55 else INK)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels([C("지역센터"), C("권역센터")], fontsize=6); ax.set_yticklabels([C("지역센터"), C("권역센터")], fontsize=6, rotation=90, va="center")
    ax.set_xlabel(tr("예측", "Predicted"), fontsize=6.5, labelpad=2); ax.set_ylabel(tr("실제", "Actual"), fontsize=6.5, labelpad=2)
    ax.set_title(title, fontsize=6.8, loc="left"); panel_label(ax, letter, x=-0.2, y=1.1)
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(length=0)
fig, axes = plt.subplots(2, 3, figsize=(W2, 4.6), gridspec_kw={"wspace": 0.45, "hspace": 0.62})
for ax, k, L in zip(axes.ravel()[:5], MODEL_NAMES, "abcde"):
    t = RES["both"][k]["test"]
    draw_cm(ax, t["TN"], t["FP"], t["FN"], t["TP"], f"{k}\nRecall {t['Recall']:.3f} | Spec. {t['Specificity']:.3f}", L)
draw_cm(axes.ravel()[5], te_opt["TN"], te_opt["FP"], te_opt["FN"], te_opt["TP"],
        f"{BEST_MODEL} (thr={THR_OPT:.2f})\nRecall {te_opt['Recall']:.3f} | Spec. {te_opt['Specificity']:.3f}", "f")
save_fig(fig, "Fig6_confusion_matrices.png")

# ---------------------------------------------------------------------------
# 5-9. 그림: Ablation (Fig7) — 점 = CV 평균, 막대 = ±SD
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(W2, 2.9), gridspec_kw={"wspace": 0.14}, sharey=True)
vlist = ["umls_only", "serial_only", "both", "none"]; off = np.linspace(-0.27, 0.27, len(vlist))
for ax, (m_, nm), L in zip(axes, [("ROC_AUC", "ROC-AUC"), ("F1", "F1-score"), ("Recall", tr("Recall (권역)", "Recall (Regional)"))], "abc"):
    for j, v in enumerate(vlist):
        mu = [RES[v][k]["cv"][m_].mean() for k in MODEL_NAMES]; sd = [RES[v][k]["cv"][m_].std() for k in MODEL_NAMES]
        ax.errorbar(mu, np.arange(len(MODEL_NAMES)) + off[j], xerr=sd, fmt="o", color=VARIANT_COL[v], markersize=2.8,
                    elinewidth=0.7, capsize=1.0, label=VARIANT_LABEL[v], markerfacecolor=VARIANT_COL[v] if v != "none" else "white")
    ax.set_xlabel(f"{nm} ({N_SPLITS}-fold CV)"); soft_grid(ax, "x"); panel_label(ax, L, x=-0.02 if L == "a" else -0.06, y=1.04)
axes[0].set_yticks(range(len(MODEL_NAMES))); axes[0].set_yticklabels(MODEL_NAMES); axes[0].invert_yaxis()
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.04), handletextpad=0.3, columnspacing=1.5)
save_fig(fig, "Fig7_ablation_chief_complaint.png")

# 해석용 요약 (최고 모델 기준 ΔAUC)
def d_auc(a, b, kind): return RES[a][kind]["cv"]["ROC_AUC"].mean() - RES[b][kind]["cv"]["ROC_AUC"].mean()
log(f"Ablation 해석({BEST_MODEL}, CV ROC-AUC): UMLS만 {RES['umls_only'][BEST_MODEL]['cv']['ROC_AUC'].mean():.4f} | "
    f"일련번호만 {RES['serial_only'][BEST_MODEL]['cv']['ROC_AUC'].mean():.4f} | 둘 다 {RES['both'][BEST_MODEL]['cv']['ROC_AUC'].mean():.4f} | "
    f"둘 다 제외 {RES['none'][BEST_MODEL]['cv']['ROC_AUC'].mean():.4f}")


# =============================================================================
# [6단계] SHAP XAI 분석 (최고 성능 부스팅 모델: CatBoost 또는 XGBoost)
#   💡 쉬운 설명: SHAP 값 = '이 변수 때문에 이 환자의 권역센터 확률(로그오즈)이 얼마나 올랐/내렸는가'.
#      양(+)이면 권역센터 쪽으로, 음(-)이면 지역센터 쪽으로 모델을 밀어 준 힘입니다.
#      (CatBoost/XGBoost 에 내장된 TreeSHAP = SHAP 라이브러리와 같은 알고리즘이며, 범주형을 원-핫 없이 그대로 설명합니다.)
# =============================================================================
banner(f"[6단계] SHAP XAI 분석 — 대상 모델: {SHAP_MODEL}")
shap_fit = RES["both"][SHAP_MODEL]["fitted"]
rng = np.random.default_rng(RANDOM_STATE)
n_s = min(SHAP_MAX_ROWS, len(X_te_all))
sel = np.sort(rng.choice(len(X_te_all), n_s, replace=False))
Xs = X_te_all.iloc[sel].reset_index(drop=True)

def compute_shap(fitted, X):
    Xp = fitted.prep(X)
    if fitted.kind == "CatBoost":
        pool = Pool(Xp, cat_features=fitted.cat_cols)
        sv = fitted.model.get_feature_importance(pool, type="ShapValues")
        raw = fitted.model.predict(pool, prediction_type="RawFormulaVal")
    else:   # XGBoost
        booster = fitted.model.get_booster(); dm = xgb.DMatrix(Xp, enable_categorical=True)
        kw = {"iteration_range": (0, int(fitted.best_iter) + 1)} if fitted.best_iter is not None else {}
        sv = booster.predict(dm, pred_contribs=True, **kw); raw = booster.predict(dm, output_margin=True, **kw)
    return sv[:, :-1], sv[:, -1], raw

SV, BASE, RAW = compute_shap(shap_fit, Xs)
SV_DF = pd.DataFrame(SV, columns=shap_fit.cols)
err = float(np.max(np.abs(SV.sum(1) + BASE - RAW)))
log(f"SHAP 계산 완료 ({n_s:,}명 × {SV.shape[1]}변수) | 정합성 검사: 최대 |ΣSHAP+기준값-모델출력| = {err:.2e} (0에 가까워야 정상)")
imp = SV_DF.abs().mean().sort_values(ascending=False)
imp_tbl = pd.DataFrame({"평균|SHAP|": imp, "비중(%)": imp / imp.sum() * 100}).round(4)
show_table(imp_tbl, "SHAP 변수 중요도 (평균 절댓값)")
save_table(imp_tbl, "Table10_shap_importance.csv")

# --- 6-1. SHAP Summary Plot (Beeswarm) ---
def beeswarm_y(v, nbins=60, width=0.42):
    v = np.asarray(v, float); n = len(v)
    if np.ptp(v) == 0: return np.zeros(n)
    q = np.round(nbins * (v - v.min()) / (np.ptp(v) + 1e-12)).astype(int)
    order = np.argsort(q + rng.random(n) * 1e-3, kind="stable")
    ys = np.zeros(n); layer, last = 0, -1
    for i in order:
        if q[i] != last: layer, last = 0, q[i]
        ys[i] = np.ceil(layer / 2) * ((layer % 2) * 2 - 1); layer += 1
    m = np.abs(ys).max()
    return ys / m * width if m > 0 else ys

TOPK = min(15, len(imp))
top_feats = list(imp.index[:TOPK])
fig, ax = plt.subplots(figsize=(W1 * 1.55, 0.27 * TOPK + 0.9))
for r, f in enumerate(top_feats):
    s = SV_DF[f].values; yy = r + beeswarm_y(s)
    if f in NUM_FEATURES:
        v = Xs[f].astype(float).values; ok = ~np.isnan(v)
        lo, hi = (np.nanpercentile(v, 5), np.nanpercentile(v, 95)) if ok.any() else (0, 1)
        nv = np.clip((v - lo) / (hi - lo + 1e-12), 0, 1)
        ax.scatter(s[~ok], yy[~ok], s=2.0, color="#c5cdd1", linewidths=0, alpha=0.55, zorder=2)   # 결측은 옅은 회색
        ax.scatter(s[ok], yy[ok], s=2.0, c=nv[ok], cmap=SHAP_CMAP, vmin=0, vmax=1, linewidths=0, alpha=0.62, zorder=3)
    else:
        ax.scatter(s, yy, s=2.0, color="#aeb8bc", linewidths=0, alpha=0.55, zorder=3)              # 범주형은 회색
ax.axvline(0, color=INK2, linewidth=0.6, zorder=1)
ax.set_yticks(range(TOPK)); ax.set_yticklabels([V(f) for f in top_feats]); ax.invert_yaxis(); ax.set_ylim(TOPK - 0.4, -0.7)
ax.set_xlabel(tr("SHAP 값 (+ → 권역센터 방향, − → 지역센터 방향)", "SHAP value (+ toward Regional EMC, − toward Local EMC)"))
# 행 단위 격자선 없이 점 분포와 축만 표시합니다.
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
sm = plt.cm.ScalarMappable(cmap=SHAP_CMAP, norm=plt.Normalize(0, 1)); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, aspect=28); cb.set_ticks([0, 1]); cb.set_ticklabels([tr("낮음", "Low"), tr("높음", "High")])
cb.set_label(tr("변수 값 (수치형) · 회색=범주형/결측", "Feature value (numeric) · gray = categorical/missing"), fontsize=5.8); cb.outline.set_linewidth(0.3); cb.ax.tick_params(labelsize=6)
ax.set_title(f"SHAP summary — {SHAP_MODEL}", loc="left", fontsize=7.5)
save_fig(fig, "Fig8_SHAP_beeswarm.png")

# --- 6-2. SHAP Feature Importance Bar Plot ---
fig, ax = plt.subplots(figsize=(W1 * 1.3, 0.2 * len(imp) + 0.9))
ordr = list(imp.index)
ax.barh(range(len(ordr)), imp.values, color=BLUE, height=0.62, edgecolor="none")
for i, v in enumerate(imp.values): ax.text(v + imp.values.max() * 0.012, i, f"{v:.3f}", va="center", fontsize=5.8, color=INK2)
ax.set_yticks(range(len(ordr))); ax.set_yticklabels([V(f) for f in ordr]); ax.invert_yaxis(); ax.set_xlim(0, imp.values.max() * 1.15)
ax.set_xlabel(tr("평균 |SHAP 값| (변수 중요도)", "Mean |SHAP value| (feature importance)")); soft_grid(ax, "x")
ax.set_title(f"SHAP feature importance — {SHAP_MODEL}", loc="left", fontsize=7.5)
save_fig(fig, "Fig9_SHAP_importance_bar.png")

# --- 6-3. SHAP Dependence Plot : 고통지속도·발병시간_범주·주증상-1·손상기전·연령 ---
def dep_numeric(ax, f, letter, log_x=False, age=False):
    x = Xs[f].astype(float).values; s = SV_DF[f].values; ok = ~np.isnan(x)
    jit = rng.normal(0, 0.12, ok.sum()) if age else 0
    xp = x[ok] + jit
    if log_x: xp = np.log10(np.clip(x[ok], FLOOR_H, None))
    ax.scatter(xp, s[ok], s=1.8, color=BLUE, alpha=0.22, linewidths=0, zorder=2)
    # 추세선: 구간별 평균 (연령은 연령군별, 그 외는 분위수 구간)
    if age:
        grp = pd.Series(s[ok]).groupby(x[ok]); xm = np.array(sorted(grp.groups)); ym = grp.mean().loc[xm].values
    else:
        q = pd.qcut(pd.Series(xp), q=min(15, max(3, ok.sum() // 150)), duplicates="drop")
        gm = pd.DataFrame({"x": xp, "s": s[ok], "q": q}).groupby("q", observed=True).mean(); xm, ym = gm["x"].values, gm["s"].values
    ax.plot(xm, ym, color=ORANGE, linewidth=1.15, zorder=4)
    ax.axhline(0, color=INK2, linewidth=0.5, linestyle="--")
    ax.set_ylabel(tr("SHAP 값", "SHAP value")); soft_grid(ax, "y"); panel_label(ax, letter, x=-0.2)
    return int((~ok).sum())

def dep_categorical(ax, f, letter, top=12, min_n=15):
    d = pd.DataFrame({"c": Xs[f].astype(str).values, "s": SV_DF[f].values})
    keep = d["c"].value_counts().head(top).index; d = d[d["c"].isin(keep)]
    g = d.groupby("c")["s"].agg(["mean", "count"]); g = g[g["count"] >= min_n].sort_values("mean")
    data = [d.loc[d["c"] == c, "s"].values for c in g.index]
    bp = ax.boxplot(data, vert=False, positions=range(len(g)), widths=0.6, patch_artist=True, showfliers=False, whis=(10, 90),
                    medianprops=dict(color=INK, linewidth=0.8), whiskerprops=dict(color=INK2, linewidth=0.6),
                    capprops=dict(color=INK2, linewidth=0.6), boxprops=dict(linewidth=0.5, edgecolor=INK2))
    for patch in bp["boxes"]: patch.set_facecolor(BLUE); patch.set_alpha(0.20)
    ax.scatter(g["mean"].values, range(len(g)), s=5, color=ORANGE, zorder=5, linewidths=0)
    nice = (lambda c: UMLS_NAME_MAP.get(c, c)) if f == UMLS else C
    ax.set_yticks(range(len(g))); ax.set_yticklabels([f"{nice(c)} (n={int(n)})" for c, n in zip(g.index, g["count"])], fontsize=5.6)
    ax.axvline(0, color=INK2, linewidth=0.5, linestyle="--"); ax.set_xlabel(tr("SHAP 값", "SHAP value")); soft_grid(ax, "x"); panel_label(ax, letter, x=-0.2 if False else -0.02)

fig = plt.figure(figsize=(W2, 7.6))
gs = fig.add_gridspec(3, 2, height_ratios=[1.0, 0.62, 1.55], hspace=0.62, wspace=0.62)
axA = fig.add_subplot(gs[0, 0]); axB = fig.add_subplot(gs[0, 1]); axC = fig.add_subplot(gs[1, 0])
axN = fig.add_subplot(gs[1, 1]); axD = fig.add_subplot(gs[2, 0]); axE = fig.add_subplot(gs[2, 1])
n_na = dep_numeric(axA, "고통지속도", "a", log_x=True)
axA.set_xticks(np.log10(tick_h)); axA.set_xticklabels(tick_lb, fontsize=5.8); axA.set_xlabel(tr("고통지속도 (로그 축)", "Onset-to-arrival time (log)"))
axA.set_xlim(np.log10(FLOOR_H), np.log10(max(np.nanpercentile(Xs["고통지속도"].astype(float), 99.5), 24)))
axA.set_title(tr(f"{V('고통지속도')} (결측 {n_na:,}명 제외)", f"{V('고통지속도')} (excl. {n_na:,} missing)"), loc="left", fontsize=7)
dep_numeric(axB, "연령", "b", age=True)
axB.set_xticks([1, 3, 5, 7, 9, 11, 13, 15, 17]); axB.set_xticklabels([AGE_LABELS[k] for k in [1, 3, 5, 7, 9, 11, 13, 15, 17]], rotation=45, ha="right", fontsize=5.6)
axB.set_xlabel(tr("연령군 (세)", "Age group (years)")); axB.set_title(V("연령"), loc="left", fontsize=7)
dep_categorical(axC, "발병시간_범주", "c"); axC.set_title(V("발병시간_범주"), loc="left", fontsize=7)
dep_categorical(axD, "손상기전", "d", top=16); axD.set_title(V("손상기전"), loc="left", fontsize=7)
dep_categorical(axE, UMLS, "e", top=14, min_n=20); axE.set_title(tr("주증상-1 (UMLS, 빈도 상위)", "Chief complaint (UMLS, most frequent)"), loc="left", fontsize=7)
axN.axis("off")
axN.text(0.0, 0.95, tr("그림 읽는 법", "How to read"), fontsize=7, fontweight="bold", va="top")
axN.text(0.0, 0.72, tr("· 점 = 환자 1명, SHAP > 0 이면 권역센터 쪽으로 기여\n· 주황 선/점 = 구간·범주별 평균\n· 상자 = 중앙값·IQR (수염 10–90 백분위)\n· 범주형은 평균 SHAP 순으로 정렬\n· SHAP 단위는 로그오즈(확률이 아님)",
                       "· Dot = one patient; SHAP > 0 favors Regional EMC\n· Orange line/dot = mean per bin/category\n· Box = median & IQR (whiskers 10–90th pct)\n· Categories sorted by mean SHAP\n· SHAP is in log-odds units (not probability)"),
         fontsize=6, va="top", color=INK2, linespacing=1.35)
save_fig(fig, "Fig10_SHAP_dependence.png")


# =============================================================================
# [7단계] 논문 작성용 핵심 요약 리포트
# =============================================================================
banner("[7단계] 논문 작성용 핵심 요약 리포트")

def fmt_ci(k, m):
    return f"{RES['both'][k]['test'][m]:.3f} (95% CI {BOOT[k][m][0]:.3f}–{BOOT[k][m][1]:.3f})"
bt = RES["both"][BEST_MODEL]["test"]
sh_top = list(imp.index[:5])
v_auc = {v: RES[v][BEST_MODEL]["cv"]["ROC_AUC"].mean() for v in ["umls_only", "serial_only", "both", "none"]}
v_rec = {v: RES[v][BEST_MODEL]["test"]["Recall"] for v in ["umls_only", "serial_only", "both", "none"]}
dur_med1, dur_med0 = da.median(), db.median()
n_total, n_pos = len(df), int(df["Y"].sum())

best_variant = max(["umls_only", "serial_only", "both"], key=lambda v: v_auc[v])
ablation_msg = {
    "both": "두 변수를 함께 넣었을 때 성능이 가장 높아, 두 변수가 서로 보완적 정보를 담고 있음을 시사한다.",
    "umls_only": "UMLS 코드만 넣었을 때가 가장 좋아, 일련번호는 추가 정보 없이 잡음으로 작용할 가능성이 있다.",
    "serial_only": "일련번호만 넣었을 때가 가장 좋아, 일련번호가 병원·기록 체계의 특성을 반영한 변수일 가능성(누수 여부 검토 필요)이 있다.",
}[best_variant]

report = f"""
=====================================================================================
 [핵심 요약 리포트]  경기도 공공차량 이송 환자 응급의료기관 종별(권역 vs 지역) 이진 분류
=====================================================================================
1. 연구 대상
   - 자료: NEDIS 응급 표본자료 2020–2024, 경기도 소재 권역·지역응급의료센터, 119구급차 및 경찰차 등 공공차량 이송 환자
   - 최종 분석 대상 {n_total:,}명 (권역 {n_pos:,}명 [{n_pos/n_total*100:.1f}%], 지역 {n_total-n_pos:,}명 [{(1-n_pos/n_total)*100:.1f}%])
   - 내원수단: 119구급차 {int(ct_cnt.loc['119구급차','합계']):,}명 / 경찰차 등 공공차량 {int(ct_cnt.loc['경찰차 등 공공차량','합계']):,}명
     (카이제곱 {pstr(p_chi)}, Cramér's V={cramers_v:.3f})
   - Train {len(idx_tr):,} / Test {len(idx_te):,} (층화 8:2), {N_SPLITS}-Fold 층화 교차검증, 시드 {RANDOM_STATE}
   - 자동 제외된 단일값 변수: {CONST_DROPPED if CONST_DROPPED else '없음'}
   - 최종 입력 변수 {len(FEATURES)}개 (원본 변수 + 파생변수 2개: 고통지속도, 발병시간_범주)

2. 파생변수 '고통지속도' (발병→내원, 시간)
   - 유효 {int(df['고통지속도'].notna().sum()):,}건 (미상코드 결측 {n_unknown:,}, 음수 {n_neg:,}, 1년 초과 {n_big:,}건은 NaN 처리)
   - 중앙값 {df['고통지속도'].median():.2f}시간 [Q1 {df['고통지속도'].quantile(.25):.2f} – Q3 {df['고통지속도'].quantile(.75):.2f}], 평균 {df['고통지속도'].mean():.2f}시간
   - 권역 중앙값 {dur_med1:.2f}h vs 지역 {dur_med0:.2f}h (Mann–Whitney {pstr(p_mw)}, P(권역>지역)={eff:.3f})

3. 모델 성능 (Test, 임계값 0.5, 95% 부트스트랩 CI)
""" + "\n".join(f"   - {k:19s}: ROC-AUC {fmt_ci(k,'ROC_AUC')} | F1 {RES['both'][k]['test']['F1']:.3f} | 권역 Recall {RES['both'][k]['test']['Recall']:.3f} | Precision {RES['both'][k]['test']['Precision']:.3f}" for k in MODEL_NAMES) + f"""
   ▶ CV ROC-AUC 기준 최고 모델: {BEST_MODEL}
      Accuracy {bt['Accuracy']:.3f}, Precision {bt['Precision']:.3f}, Recall(민감도) {bt['Recall']:.3f}, Specificity {bt['Specificity']:.3f},
      F1 {bt['F1']:.3f}, ROC-AUC {fmt_ci(BEST_MODEL,'ROC_AUC')}, PR-AUC {fmt_ci(BEST_MODEL,'PR_AUC')}
      혼동행렬: TN {bt['TN']:,} / FP {bt['FP']:,} / FN {bt['FN']:,} / TP {bt['TP']:,}
      최적 임계값(OOF F1 최대) {THR_OPT:.2f} 적용 시: Recall {te_opt['Recall']:.3f}, Precision {te_opt['Precision']:.3f}, F1 {te_opt['F1']:.3f}
   ▶ 해석: FP={bt['FP']:,}명은 '지역센터로 충분한데 권역센터로 안내된' 환자(권역 과밀·불필요한 장거리 이동 후보),
           FN={bt['FN']:,}명은 '권역센터가 필요한데 지역센터로 안내된' 환자(전원 위험 후보)이다.

4. Ablation (주증상 변수 구성, {BEST_MODEL}, CV ROC-AUC)
   - UMLS만 {v_auc['umls_only']:.4f} | 일련번호만 {v_auc['serial_only']:.4f} | 둘 다 {v_auc['both']:.4f} | (참고) 둘 다 제외 {v_auc['none']:.4f}
   - Test 권역 Recall: UMLS만 {v_rec['umls_only']:.3f} | 일련번호만 {v_rec['serial_only']:.3f} | 둘 다 {v_rec['both']:.3f} | 둘 다 제외 {v_rec['none']:.3f}
   - 해석: {ablation_msg}

5. SHAP ({SHAP_MODEL}, Test {n_s:,}명 표본) 상위 5개 변수
""" + "\n".join(f"   {i+1}. {f}  (평균|SHAP| {imp[f]:.3f}, 비중 {imp[f]/imp.sum()*100:.1f}%)" for i, f in enumerate(sh_top)) + f"""

6. 논문 Methods/Results 초안 문장 (수치 자동 삽입, 투고 전 반드시 본인 검토)
   [Methods] 2020–2024년 NEDIS 표본자료 중 경기도 권역·지역응급의료센터에 119구급차 또는 경찰차 등 공공차량으로 내원한 {n_total:,}명을 분석하였다.
   종속변수는 응급의료기관 종별(권역=1, 지역=0)이며, 발병–내원 소요시간(고통지속도)과 발병시간대 범주를 파생하였다.
   데이터를 층화 8:2로 분할하고 학습 데이터에서 5-fold 교차검증을 수행하였으며, 클래스 불균형은 가중치(class weight/scale_pos_weight)로 보정하였다.
   로지스틱 회귀, 랜덤포레스트, LightGBM, XGBoost, CatBoost를 비교하였고, 성능은 ROC-AUC, PR-AUC, F1, 권역센터 재현율 등으로 평가하였다.
   [Results] {BEST_MODEL} 모델은 검증 세트에서 ROC-AUC {bt['ROC_AUC']:.3f}, PR-AUC {bt['PR_AUC']:.3f}, 권역센터 재현율 {bt['Recall']:.3f}, F1 {bt['F1']:.3f}을 보였다.
   SHAP 분석 결과 {', '.join(sh_top[:3])} 순으로 모델 예측에 기여하였다.

7. 한계점 및 주의사항 (논문 Discussion에 반드시 반영 권장)
   - 타겟은 '실제 이송된 기관 종별'이며 '임상적으로 최적이었던 기관'이 아니다 (이송 결정의 관행·병상 여건이 반영됨).
   - 내원경로·응급증상 해당 여부·주증상 일련번호 등은 병원 도착 후 기록되는 변수일 수 있어, '현장 단계 즉시 분류'에 쓰려면
     현장에서 실제로 확보 가능한 변수만으로 재학습한 모델(Ablation의 '둘 다 제외' 등)과 비교·보완 검증이 필요하다.
   - 거리·이동시간·병상 정보가 없어 '이동 시간·거리 절감 효과'는 직접 추정하지 못했다 (FP/FN 수로 간접 해석). 후속 연구에서 위치 데이터 결합 필요.
   - 표본자료 기반 단일 지역·후향적 분석이며 외부 검증 코호트가 없다. 내원일자(경과일)가 병원 지정·운영 변화의 시간 추세를 반영할 수 있다.
   - 발병시간의 '1111'은 미상 코드로 처리하여 실제 11시 11분 발병도 함께 결측 처리된다(규모 미미).
=====================================================================================
"""
print(report)
rep_path = os.path.join(OUTPUT_DIR, "Summary_report.txt")
with open(rep_path, "w", encoding="utf-8") as fh:
    fh.write(report)

# 모든 표를 하나의 엑셀로 묶어 저장 (논문 작성 편의)
try:
    xlsx_path = os.path.join(OUTPUT_DIR, "All_tables.xlsx")
    with pd.ExcelWriter(xlsx_path) as xw:
        for nm, t_, ix in [("Table0_funnel", funnel_df, False), ("Table0_quality", dq, True), ("Table1_baseline", table1, False),
                           ("Table2_target_x_mode", ct_show, True), ("Table3_duration", dur_stats, True),
                           ("Table6_cv", cv_tbl, True), ("Table7_test", test_tbl.round(4), True), ("Table8_bootCI", ci_tbl, True),
                           ("Table9_ablation", abl_tbl, False), ("Table9_abl_wide", abl_summary, True), ("Table10_shap", imp_tbl, True)]:
            t_.to_excel(xw, sheet_name=nm[:31], index=ix)
    log(f"표 모음 저장: {xlsx_path}")
except Exception as e:
    log(f"(엑셀 묶음 저장 건너뜀: {e})")

log("저장된 그림 목록:")
for p_ in FIG_LIST: print("   ·", p_)
log(f"✅ 전체 완료! 결과 폴더: {os.path.abspath(OUTPUT_DIR)}  (총 {(time.time()-T0)/60:.1f}분)")
